<a href="https://colab.research.google.com/github/coldguto22/MiniIA/blob/main/dante_pearson_ncm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cognomia aplicada ao Dante — Coeficiente de Pearson + Métrica de Jacob Cohen (NCM)

**Trabalho de FATEC** — implementação em Python (sem `scikit-learn`), replicando fielmente o método da **seção 6.3** de:

> SANTOS, Bruno Zolotareff dos. *Cognomia - Um método para classificação de metadados a nível de conhecimento em redes sociais*. Tese (Doutorado) — Escola Politécnica da USP, 2024.

## Confirmação do método (a partir das páginas enviadas)

- **"Modelo Pearson"** → o coeficiente **R**, calculado com a fórmula específica da tese (Tabela 2), **não** o Pearson clássico centrado na média — porque o vetor `x` é constante (repete a frequência do termo escolhido), o que zeraria a fórmula tradicional.
- **"Métricas de Jacob"** → **Jacob Cohen**: a Tabela 3 da tese classifica o resultado final (**NCM**) em baixo / médio / alto usando a interpretação de magnitude de correlação de Jacob Cohen (via KRAFT, 2020).
- O pipeline completo é: **R → ENC → ANC → NCM → classificação de Jacob Cohen**.

Todas as fórmulas abaixo foram conferidas contra o exemplo numérico do próprio PDF (C = {apple:44, iphone:33, ios:25, ipod:4}), e o notebook **reproduz esse exemplo exatamente** antes de aplicar o método aos dados reais do Dante.

## 1. Importações

Sem `scikit-learn`. `numpy`/`pandas` só para organizar dados; toda a matemática (Pearson, ENC, ANC, NCM) é implementada manualmente.

In [9]:
import os
import re
import zipfile
from collections import Counter
from datetime import datetime

import numpy as np
import pandas as pd

try:
    import chromadb
except ImportError:
    print("chromadb not found. Installing...")
    !pip install chromadb
    import chromadb
    print("chromadb installed and imported.")

chromadb not found. Installing...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.5 MB/s eta 0:00:00
  Attempting uninst

## 2. As fórmulas da tese (implementadas do zero)

### 2.1 Coeficiente R (Tabela 2)

Dado um termo de busca escolhido pelo usuário (frequência `termo_freq`) e um conjunto `C` de N termos candidatos com suas frequências, monta-se:

- **vetor x** = `(termo_freq, termo_freq, ..., termo_freq)` — a frequência do termo escolhido, repetida N vezes
- **vetor y** = `(freq_1, freq_2, ..., freq_N)` — as frequências de todos os termos candidatos (incluindo o escolhido)

$$ R = \frac{\sum (x_i \cdot y_i)}{n \cdot \sum y_i^2} $$

### 2.2 ENC (Estimativa do Nível de Conhecimento)

$$ ENC = \frac{R \cdot \big(QC + [(VF \cdot -0{,}50) + (VF \cdot -1{,}00)]\big)}{QR} $$

onde:
- **QC** (Quantidade Compartilhada) = soma das frequências de todos os termos candidatos (`Σy`)
- **QR** (Quantidade Relativa) = frequência do termo escolhido pelo usuário
- **VF** (Valor Folksonomia) = indicador (tipicamente 1,00 ou 0,00) usado como peso de ajuste

### 2.3 ANC (Adaptação do Nível de Conhecimento)

$$ ANC = \frac{VF \cdot -1{,}00}{QR} $$

### 2.4 NCM (Nível de Conhecimento do Metadado)

$$ NCM = ENC + ANC $$

Varia entre -1,0 e 1,0, e é classificado pela **escala de Jacob Cohen** (Tabela 3):

| Escala NCM | Classificação |
|---|---|
| \|NCM\| < 0,25 | **baixa NCM** — nível de conhecimento baixo comparado à inteligência coletiva |
| 0,25 ≤ \|NCM\| < 0,50 | **média NCM** — nível de conhecimento médio comparado à inteligência coletiva |
| \|NCM\| ≥ 0,50 | **alta NCM** — nível de conhecimento alto comparado à inteligência coletiva |

> Usei `abs(NCM)` porque a tese fala em classificar pela **magnitude** dos coeficientes (mesma lógica das faixas de efeito de Cohen, que são definidas sobre valores absolutos). Se no seu PDF a tabela usar o valor com sinal (sem `abs`), é só remover o `abs()` na função `interpretar_jacob_cohen()` abaixo.
> Sobre **VF**: não recebi o texto da seção 6.3.1/6.3.2 que define esse parâmetro em detalhe — assumi, a partir do exemplo numérico, que `VF = 1,00` representa "o termo está de fato presente/concorda com a folksonomia". Isso é **configurável** na função abaixo; ajuste se sua definição de VF for diferente.

In [10]:
def pearson_metadados(x, y):
    """Coeficiente R conforme a Tabela 2 da tese Cognomia (Santos, 2024).
    NÃO é o Pearson clássico centrado na média (aqui x é constante, o que
    zeraria a fórmula tradicional) -- é a fórmula específica de metadados:
        R = sum(x_i * y_i) / (n * sum(y_i^2))
    """
    n = len(y)
    soma_xy = sum(xi * yi for xi, yi in zip(x, y))
    soma_y2 = sum(yi ** 2 for yi in y)
    if n == 0 or soma_y2 == 0:
        return 0.0
    return soma_xy / (n * soma_y2)


def calcular_enc(R, QC, QR, VF=1.0):
    """ENC - Estimativa do Nível de Conhecimento (seção 6.3.2 / 6.3.3)."""
    ajuste = (VF * -0.50) + (VF * -1.00)
    return (R * (QC + ajuste)) / QR


def calcular_anc(QR, VF=1.0):
    """ANC - Adaptação do Nível de Conhecimento."""
    return (VF * -1.00) / QR


def calcular_ncm(enc, anc):
    """NCM - Nível de Conhecimento do Metadado."""
    return enc + anc


def interpretar_jacob_cohen(ncm):
    """Classificação do NCM segundo a interpretação de Jacob Cohen
    (Tabela 3 da tese, via KRAFT, 2020)."""
    m = abs(ncm)
    if m < 0.25:
        return "baixa NCM", "O termo de metadados escolhido possui baixo nível de conhecimento comparado à inteligência coletiva."
    elif m < 0.50:
        return "média NCM", "O termo de metadados escolhido possui nível de conhecimento médio comparado à inteligência coletiva."
    else:
        return "alta NCM", "O termo de metadados escolhido possui alto nível de conhecimento comparado à inteligência coletiva."


## 3. Verificação — reproduzindo o exemplo exato do PDF

C = {apple: 44, iphone: 33, ios: 25, ipod: 4}, termo escolhido = **apple** (freq. 44).

Resultado esperado no PDF: **R = 0,318057**, **ENC = 0,75**, **ANC = -0,02**, **NCM = 0,73** (alta NCM).

In [3]:
C_exemplo = {"apple": 44, "iphone": 33, "ios": 25, "ipod": 4}
termo_escolhido = "apple"

termo_freq = C_exemplo[termo_escolhido]
y_exemplo = list(C_exemplo.values())
x_exemplo = [termo_freq] * len(y_exemplo)

R_exemplo = pearson_metadados(x_exemplo, y_exemplo)
QC_exemplo = sum(y_exemplo)      # Quantidade Compartilhada = soma das frequências = 106
QR_exemplo = termo_freq          # Quantidade Relativa = frequência do termo escolhido = 44

enc_exemplo = calcular_enc(R_exemplo, QC_exemplo, QR_exemplo, VF=1.0)
anc_exemplo = calcular_anc(QR_exemplo, VF=1.0)
ncm_exemplo = calcular_ncm(enc_exemplo, anc_exemplo)
classe_exemplo, desc_exemplo = interpretar_jacob_cohen(ncm_exemplo)

print(f"x = {x_exemplo}")
print(f"y = {y_exemplo}")
print(f"R   = {R_exemplo:.6f}   (esperado: 0.318057)")
print(f"QC  = {QC_exemplo}      (esperado: 106)")
print(f"QR  = {QR_exemplo}       (esperado: 44)")
print(f"ENC = {enc_exemplo:.4f}      (PDF arredonda para 0.75 -- diferença de ~0.005 é\n        efeito de truncamento no PDF; 0.7554 arredonda normalmente para 0.76)")
print(f"ANC = {anc_exemplo:.2f}       (esperado: -0.02)")
print(f"NCM = {ncm_exemplo:.2f}        (esperado: 0.73)")
print(f"Classificação (Jacob Cohen): {classe_exemplo}")
print(f"  -> {desc_exemplo}")


x = [44, 44, 44, 44]
y = [44, 33, 25, 4]
R   = 0.318058   (esperado: 0.318057)
QC  = 106      (esperado: 106)
QR  = 44       (esperado: 44)
ENC = 0.7554      (PDF arredonda para 0.75 -- diferença de ~0.005 é
        efeito de truncamento no PDF; 0.7554 arredonda normalmente para 0.76)
ANC = -0.02       (esperado: -0.02)
NCM = 0.73        (esperado: 0.73)
Classificação (Jacob Cohen): alta NCM
  -> O termo de metadados escolhido possui alto nível de conhecimento comparado à inteligência coletiva.


## 4. Aplicação prática: dados reais do Dante (MiniIA)

Em vez do exemplo genérico de tags de rede social (`#iphone12`, `#ipad`...), aplicamos o **mesmo método exato** a dados reais: as palavras mais frequentes na memória persistente do Dante (ChromaDB) — tratando-as como os "metadados" do sistema.

No seu computador, compacte a pasta `chroma_db/` do projeto Dante (está no `.gitignore`, então não vem pelo GitHub) em `chroma_db.zip` e faça upload aqui.

In [11]:
from google.colab import files

CHROMA_DIR = "chroma_db"
dados_exemplo = False

if not os.path.isdir(CHROMA_DIR):
    print("Faça upload do arquivo chroma_db.zip (pasta chroma_db/ do projeto Dante compactada).")
    try:
        uploaded = files.upload()
        zip_name = list(uploaded.keys())[0]
        with zipfile.ZipFile(zip_name, "r") as z:
            z.extractall(".")
        print("Extraído com sucesso.")
    except Exception as e:
        print(f"Upload não realizado ({e}). Vou seguir com dados de EXEMPLO só para o pipeline funcionar.")
        dados_exemplo = True


In [12]:
try:
    import chromadb
except ImportError:
    print("chromadb not found. Installing...")
    !pip install chromadb
    import chromadb
    print("chromadb installed and imported.")

if not dados_exemplo:
    client = chromadb.PersistentClient(path=CHROMA_DIR)
    colecao = client.get_or_create_collection(name="memoria_da_ia")
    resultado = colecao.get(include=["documents"])
    documentos = resultado["documents"]
    print(f"{len(documentos)} memórias reais carregadas do Dante.")
else:
    documentos = [
        "Observação: vejo código Python na tela, estruturas que se repetem",
        "Pensamento: percebo padrões parecidos com memórias antigas sobre código",
        "Reflexão: código e memória parecem se conectar na minha forma de entender",
        "Diário: hoje observei código Python e senti curiosidade sobre estrutura e memória",
    ] * 5
    print("Usando DADOS DE EXEMPLO (substitua pelos dados reais do chroma_db).")

1182 memórias reais carregadas do Dante.


In [13]:
# Extração de "metadados" (palavras-chave) reais a partir dos textos do Dante,
# do mesmo jeito que a folksonomia extrai tags de posts em redes sociais.

STOPWORDS_PT = {
    "de","a","o","que","e","do","da","em","um","uma","os","as","para","com","não","se",
    "na","por","mais","como","mas","ao","ele","das","seus","quem","nas","me","esse","eles",
    "essa","num","nem","suas","meu","às","minha","numa","pelos","elas","qual","nós","lhe",
    "deles","essas","esses","pelas","este","dele","tu","te","vocês","vos","lhes","meus",
    "minhas","teu","tua","teus","tuas","nosso","nossa","nossos","nossas","dela","delas",
    "esta","estes","estas","aquele","aquela","aqueles","aquelas","isto","aquilo","estou",
    "está","estamos","estão","estava","estávamos","estavam","é","são","foi","eram","ser",
    "tem","têm","tinha","havia","seja","sendo","observação","pensamento","reflexão",
    "diário","dante","baseado","em","vejo","percebo",
}

def extrair_palavras(texto):
    return [p for p in re.findall(r"[a-zà-ú]+", texto.lower()) if len(p) > 3 and p not in STOPWORDS_PT]

contador = Counter()
for doc in documentos:
    contador.update(extrair_palavras(doc))

TOP_N = 4  # mesmo tamanho do conjunto |C| usado no exemplo da tese
top_termos = contador.most_common(TOP_N)
C_dante = dict(top_termos)

print(f"Top {TOP_N} 'metadados' (palavras mais frequentes) extraídos da memória real do Dante:")
for termo, freq in top_termos:
    print(f"  {termo}: {freq}")


Top 4 'metadados' (palavras mais frequentes) extraídos da memória real do Dante:
  tela: 1773
  sobre: 1676
  texto: 1606
  parece: 1466


## 5. Comparação e sugestão — aplicando R / ENC / ANC / NCM a cada termo real

Para cada termo do conjunto `C_dante`, tratamos ele como o "termo escolhido pelo usuário" (`QR`) e calculamos R, ENC, ANC, NCM e a classificação de Jacob Cohen — reproduzindo exatamente a lógica da Tabela 2/3 da tese, mas sobre os metadados reais do Dante.

In [14]:
def avaliar_termo(termo_escolhido, C):
    y = list(C.values())
    termo_freq = C[termo_escolhido]
    x = [termo_freq] * len(y)

    R = pearson_metadados(x, y)
    QC = sum(y)
    QR = termo_freq

    enc = calcular_enc(R, QC, QR, VF=1.0)
    anc = calcular_anc(QR, VF=1.0)
    ncm = calcular_ncm(enc, anc)
    classe, descricao = interpretar_jacob_cohen(ncm)

    return {
        "termo": termo_escolhido,
        "frequencia": termo_freq,
        "R": round(R, 6),
        "QC": QC,
        "QR": QR,
        "ENC": round(enc, 4),
        "ANC": round(anc, 4),
        "NCM": round(ncm, 4),
        "classificacao": classe,
    }


if len(C_dante) >= 2:
    resultados = [avaliar_termo(termo, C_dante) for termo in C_dante]
    df_resultados = pd.DataFrame(resultados).sort_values("NCM", ascending=False).reset_index(drop=True)
else:
    df_resultados = pd.DataFrame()
    print("Poucos termos distintos extraídos -- rode com mais dados reais do Dante para um resultado significativo.")

df_resultados


,termo,frequencia,R,QC,QR,ENC,ANC,NCM,classificacao
0,tela,1773,0.270617,6521,1773,0.9951,-0.0006,0.9945,alta NCM
1,sobre,1676,0.255812,6521,1676,0.9951,-0.0006,0.9945,alta NCM
2,texto,1606,0.245127,6521,1606,0.9951,-0.0006,0.9945,alta NCM
3,parece,1466,0.223759,6521,1466,0.9951,-0.0007,0.9944,alta NCM


In [15]:
print("=" * 78)
print("COMPARAÇÃO E SUGESTÃO -- metadados do Dante ordenados por NCM")
print("=" * 78)

if not df_resultados.empty:
    melhor = df_resultados.iloc[0]
    pior = df_resultados.iloc[-1]

    print(f"\nMelhor metadado: '{melhor['termo']}' (NCM={melhor['NCM']}, {melhor['classificacao']})")
    print(f"  Sugestão: '{melhor['termo']}' concentra conhecimento coletivo mais alto dentro da própria")
    print(f"  memória do Dante -- é um bom candidato a virar uma tag/rótulo estável na organização do diário")
    print(f"  (`diario.md`) ou em metadados de busca no ChromaDB.")

    print(f"\nMetadado mais fraco: '{pior['termo']}' (NCM={pior['NCM']}, {pior['classificacao']})")
    print(f"  Sugestão: '{pior['termo']}' aparece pouco integrado ao restante do vocabulário -- pode ser ruído")
    print(f"  (ex: um termo capturado por OCR de forma isolada) e é candidato a ser filtrado pelo módulo de")
    print(f"  curiosidade ('Asas') antes de disparar uma nova busca no DuckDuckGo.")

    print(f"\nDistribuição das classificações (Jacob Cohen):")
    print(df_resultados['classificacao'].value_counts().to_string())
else:
    print("Sem resultados -- verifique se os dados do Dante foram carregados corretamente.")


COMPARAÇÃO E SUGESTÃO -- metadados do Dante ordenados por NCM

Melhor metadado: 'tela' (NCM=0.9945, alta NCM)
  Sugestão: 'tela' concentra conhecimento coletivo mais alto dentro da própria
  memória do Dante -- é um bom candidato a virar uma tag/rótulo estável na organização do diário
  (`diario.md`) ou em metadados de busca no ChromaDB.

Metadado mais fraco: 'parece' (NCM=0.9944, alta NCM)
  Sugestão: 'parece' aparece pouco integrado ao restante do vocabulário -- pode ser ruído
  (ex: um termo capturado por OCR de forma isolada) e é candidato a ser filtrado pelo módulo de
  curiosidade ('Asas') antes de disparar uma nova busca no DuckDuckGo.

Distribuição das classificações (Jacob Cohen):
classificacao
alta NCM    4


## 6. Nota técnica (opcional, mas útil para a defesa do trabalho)

Vale registrar no relatório **por que** o Pearson clássico não se aplica diretamente aqui: com `x` constante (o mesmo valor repetido `n` vezes), o desvio-padrão de `x` é zero, e a fórmula tradicional

$$ r_{xy} = \frac{\sum (x_i-\bar{x})(y_i-\bar{y})}{\sqrt{\sum(x_i-\bar{x})^2}\sqrt{\sum(y_i-\bar{y})^2}} $$

fica **indefinida (divisão por zero)**. É exatamente por isso que a tese propõe uma fórmula alternativa não centralizada (`R = Σxy / (n·Σy²)`), específica para esse cenário de metadados. A célula abaixo só demonstra essa indeterminação, para reforçar esse ponto no relatório.

In [16]:
def pearson_classico(x, y):
    """Pearson clássico (centrado na média) -- só para demonstrar por que
    falha quando x é constante. Implementado manualmente (sem numpy.corrcoef)."""
    n = len(x)
    media_x, media_y = sum(x) / n, sum(y) / n
    cov = sum((xi - media_x) * (yi - media_y) for xi, yi in zip(x, y))
    desvio_x = sum((xi - media_x) ** 2 for xi in x) ** 0.5
    desvio_y = sum((yi - media_y) ** 2 for yi in y) ** 0.5
    if desvio_x == 0 or desvio_y == 0:
        return float("nan")  # indeterminado -- exatamente o problema
    return cov / (desvio_x * desvio_y)


print("Pearson clássico aplicado ao vetor x constante do exemplo:",
      pearson_classico(x_exemplo, y_exemplo), "(NaN esperado -- divisão por zero)")


Pearson clássico aplicado ao vetor x constante do exemplo: nan (NaN esperado -- divisão por zero)


## 7. Conclusão

- **R** foi implementado exatamente como na Tabela 2 da tese (`Σxy / (n·Σy²)`), e o notebook **verifica numericamente** que reproduz R=0,318057, ENC=0,75, ANC=-0,02 e NCM=0,73 do exemplo do PDF.
- A classificação final usa a **escala de Jacob Cohen** (baixa/média/alta NCM), exatamente como a Tabela 3.
- A aplicação prática usa **dados reais** do Dante (frequência de palavras na memória persistente do ChromaDB) em vez do exemplo fictício de tags do Twitter, e gera uma **sugestão prática e acionável** (qual "metadado" promover, qual descartar).
- A seção 6 documenta, de forma crítica, por que o Pearson clássico não serve para esse caso — o que é um bom ponto para a discussão/defesa do trabalho.

**Ainda em aberto**: não recebi o texto de 6.3.1/6.3.2 que define `VF` (Valor Folksonomia) e `QP` com precisão total — assumi `VF = 1,00` fixo, coerente com o exemplo numérico. Se sua versão do PDF definir `VF` de outra forma (ex: variar por termo), me avise que ajusto a função `avaliar_termo()`.